## Time-weighted Vector Retriever

 - 가장 최근에 이용된 문서를 기준으로 먼저 참고하도록 하여 답변의 최신화를 유지할 수 있음
 - 시간이 지난 만큼 페널티를 부여!

In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [ ]:
!pip install langchain==0.1.16

In [ ]:
!pip install faiss-gpu-cu12

In [11]:
from langchain.embeddings import HuggingFaceEmbeddings

model_name = "jhgan/ko-sbert-nli"
encode_kwargs = {'normalize_embeddings': True}
ko_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs = encode_kwargs
)

In [12]:
from datetime import datetime, timedelta
import numpy
import faiss
from langchain.docstore import InMemoryDocstore
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain.schema import Document
from langchain_community.vectorstores import FAISS

In [13]:
embedding_size = 768
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(ko_embedding, index, InMemoryDocstore({}), {})
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore, decay_rate=0.99, k=1
)

In [14]:
yesterday = datetime.now() - timedelta(days=1)
retriever.add_documents(
    [Document(page_content="영어는 훌륭합니다.", metadata={"last_accessed_at": yesterday})]
)
retriever.add_documents([Document(page_content="한국어는 훌륭합니다")])

['73d033d6-0cb2-4e84-9c15-5a4d069793c1']

In [15]:
retriever.get_relevant_documents("영어가 어려워요")

/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores.py:330: UserWarning: Relevance scores must be between 0 and 1, got [(Document(page_content='영어는 훌륭합니다.', metadata={'last_accessed_at': datetime.datetime(2026, 1, 18, 8, 5, 16, 347474), 'created_at': datetime.datetime(2026, 1, 19, 8, 5, 16, 347746), 'buffer_idx': 0}), np.float32(0.15314615)), (Document(page_content='한국어는 훌륭합니다', metadata={'last_accessed_at': datetime.datetime(2026, 1, 19, 8, 5, 16, 459752), 'created_at': datetime.datetime(2026, 1, 19, 8, 5, 16, 459752), 'buffer_idx': 1}), np.float32(-0.11495972))]
  warnings.warn(


[Document(page_content='한국어는 훌륭합니다', metadata={'last_accessed_at': datetime.datetime(2026, 1, 19, 8, 5, 16, 895465), 'created_at': datetime.datetime(2026, 1, 19, 8, 5, 16, 459752), 'buffer_idx': 1})]